# Rf prediction
Input: `./DB/Rf/pred_example.csv` format csv  
(Required:`Smiles, elu1_smiles, elu2_smiles, ratio1, ratio2; ratio: **volume ratio**)


In [16]:
from rdkit import Chem

import numpy as np
import pandas as pd
import torch

from torch_geometric.data import Data
from torch_geometric.data import InMemoryDataset
from torch_geometric.loader import DataLoader

import torch.nn as nn
import torch.nn.functional as F
import torch_geometric.nn as pyg_nn
from tqdm.auto import tqdm

from rdkit.Chem import Descriptors
import os
import glob
import matplotlib.pyplot as plt

In [17]:
def inverse_standardization(pred_nd_array, mean, std):
    return pred_nd_array * std + mean

In [18]:
# ratio conversion 

def vol_2_mol_ratio(df, elu1_col='elu1_smiles', elu2_col='elu2_smiles'):
    df = df.copy()
    elu_inform = pd.read_csv('./DB/data_process/eluent_standard.csv')
    d_converter = dict(zip(elu_inform['hex_Smiles'], elu_inform['density'].astype(float)))

    for col in [elu1_col, elu2_col]:
        unknown = set(df[col].dropna()) - set(d_converter)
        if unknown:
            raise ValueError(f'The density is missing in eluent_standard.csv: {unknown}')

    mol1 = [r * d_converter[s] / Descriptors.MolWt(Chem.MolFromSmiles(s)) if pd.notna(s) else r
            for s, r in zip(df[elu1_col], df['ratio1'])]
    mol2 = [r * d_converter[s] / Descriptors.MolWt(Chem.MolFromSmiles(s)) if pd.notna(s) else r
            for s, r in zip(df[elu2_col], df['ratio2'])]
    mol1, mol2 = np.array(mol1), np.array(mol2)

    # mol ratio normalization
    ratio_total = mol1 + mol2
    df['ratio1'] = np.round(mol1 / ratio_total, 6)
    df['ratio2'] = np.round(mol2 / ratio_total, 6)
    return df

## MODEL selection (BLI_GAT)

In [19]:
from GAT_model_architecture import GAT_BLI

path = glob.glob('ckpt_paper/ckpt_*C_paper_final*tvt*_GAT_BLI_dropout0.3')[0]

# path

In [20]:
def make_graph_predict(model, ckpt, csv_name):

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    opt_ckpt = torch.load((ckpt + '/opt_checkpoint.pt'), map_location=device)
    model.load_state_dict(opt_ckpt['model_state_dict'])


    mean = np.load('./standardization/mean.npy')
    std = np.load('./standardization/std.npy')


    # ====================================================
    from make_graph_dataloader import MolToGraph
    from make_graph_dataloader import TripleGraphDataset
    path_raw = './DB/Rf'
    print('>>> making graph...')

    for suffix in ['compound', 'elu1', 'elu2']:
        cached = os.path.join(path_raw, 'processed', 'experimental', f'{csv_name}_{suffix}.pt')
        if os.path.exists(cached):
            os.remove(cached)

    _org_np_load = np.load
    _old_root = '/home/ufslab223/Shared/KSW/torch_geometric/TLC_RF/'

    def _local_np_load(file, *args, **kwargs):
        if isinstance(file, str) and file.startswith(_old_root) and not os.path.exists(file):
            file = os.path.join(os.getcwd(), file[len(_old_root):])
        return _org_np_load(file, *args, **kwargs)
    np.load = _local_np_load
    try:
        compound_trend_set = MolToGraph('Smiles', path_raw, csv_name, f'experimental/{csv_name}_compound.pt', molecular_param = False, augment = True)
        elu1_trend_set = MolToGraph('elu1_smiles', path_raw, csv_name, f'experimental/{csv_name}_elu1.pt', molecular_param = False, augment = True)
        elu2_trend_set = MolToGraph('elu2_smiles', path_raw, csv_name, f'experimental/{csv_name}_elu2.pt', molecular_param = False, augment = True)
    finally:
        np.load = _org_np_load

    # ====================================================
    print('>>> collecting: compound, eluent graph...')
    graph_trend_set = TripleGraphDataset(
        compound_trend_set,
        elu1_trend_set,
        elu2_trend_set
    )

    #=====================================================
    print('>>> making: dataloader...')
    trend_loader = DataLoader(graph_trend_set, batch_size =500, shuffle=False)

    # ====================================================
    print('>>> predicting...')

    trend_pred=[]
    trend_id=[]
    model.eval()
    with torch.no_grad():
        for com_trend, elu1_trend, elu2_trend in trend_loader:
            trend_id.append(com_trend.sample_id)
            com_trend_batch = com_trend.to(device)
            elu1_trend_batch = elu1_trend.to(device)
            elu2_trend_batch = elu2_trend.to(device)
            pred = model(com_trend_batch, elu1_trend_batch, elu2_trend_batch)
            trend_pred.append(pred.cpu())

    trend_pred = np.vstack(trend_pred)
    trend_pred = inverse_standardization(trend_pred, mean, std)
    trend_pred = trend_pred.reshape(-1)
    print('>> 예측 완료')
    return trend_pred

In [26]:
def predict_vol_ratio_csv(input_csv, output_csv=None):
    # Input: ./DB/Rf/pred_example.csv format csv
    # (Required:`Smiles, elu1_smiles, elu2_smiles, ratio1, ratio2; ratio: volume ratio)
    input_df = pd.read_csv(input_csv)

    required_columns = ['Smiles', 'elu1_smiles', 'elu2_smiles', 'ratio1', 'ratio2']
    missing_columns = [column for column in required_columns if column not in input_df.columns]
    if missing_columns:
        raise ValueError(f'missing column for an Input file: {missing_columns}')

    # vol/vol to mol/mol
    model_input = vol_2_mol_ratio(input_df[required_columns + (['Rf'] if 'Rf' in input_df.columns else [])])
    if 'Rf' not in model_input.columns:
        model_input['Rf'] = np.nan
    model_input['sample_id'] = np.arange(len(model_input))

    name = os.path.splitext(os.path.basename(input_csv))[0]
    model_input_name = f'{name}_molfrac_input.csv'
    model_input.to_csv(f'./DB/Rf/{model_input_name}', index=False)

    predict = make_graph_predict(model=GAT_BLI(), ckpt=path, csv_name=model_input_name)
    predict = np.clip(predict, 0, 1)
    if len(predict) != len(input_df):
        raise ValueError(f'input column #({len(input_df)}) and prediction column # ({len(predict)})is different.')

    result = input_df.copy()
    # result['mol_ratio1'] = model_input['ratio1'].values
    # result['mol_ratio2'] = model_input['ratio2'].values
    result['BLI_GAT_Rf'] = predict

    if output_csv is None:
        output_csv = f'./DB/Rf/{name}_molfrac_predicted.csv'
    result.to_csv(output_csv, index=False)
    print(f'>> save: {output_csv}')
    return result

## prediction

In [27]:
input_csv = './DB/Rf/pred_list.csv'
result = predict_vol_ratio_csv(input_csv, output_csv='./DB/Rf/pred_result.csv')
result.head(15)

>>> making graph...


Processing...


  0%|          | 0/198 [00:00<?, ?it/s]

Done!
Processing...


  0%|          | 0/198 [00:00<?, ?it/s]

Done!
Processing...


  0%|          | 0/198 [00:00<?, ?it/s]

Done!


>>> collecting: compound, eluent graph...
>>> making: dataloader...
>>> predicting...
>> 예측 완료
>> save: ./DB/Rf/pred_result.csv


,Smiles,elu1_smiles,elu2_smiles,ratio1,ratio2,Rf,BLI_GAT_Rf
0,C1=CC=C(C=C1)N,CCCCCC,CCOC(C)=O,1.0,0.0,0.04,0.011619
1,C1=CC=C(C=C1)N,CCCCCC,CCOC(C)=O,0.9,0.1,0.19,0.086426
2,C1=CC=C(C=C1)N,CCCCCC,CCOC(C)=O,0.8,0.2,0.49,0.205202
3,C1=CC=C(C=C1)N,CCCCCC,CCOC(C)=O,0.7,0.3,0.62,0.430137
4,C1=CC=C(C=C1)N,CCCCCC,CCOC(C)=O,0.6,0.4,0.67,0.596444
5,C1=CC=C(C=C1)N,CCCCCC,CCOC(C)=O,0.5,0.5,0.73,0.720846
6,C1=CC=C(C=C1)N,CCCCCC,CCOC(C)=O,0.4,0.6,0.85,0.798331
7,C1=CC=C(C=C1)N,CCCCCC,CCOC(C)=O,0.3,0.7,0.85,0.841115
8,C1=CC=C(C=C1)N,CCCCCC,CCOC(C)=O,0.2,0.8,0.85,0.880773
9,C1=CC=C(C=C1)N,CCCCCC,CCOC(C)=O,0.1,0.9,0.92,0.916921
